# Gated DistilBERT + CLIP fusion — independent experiment
Run sequentially in a **Colab GPU runtime**. Reuse the existing 3,000/500/500 pairs and feature shards. No dataset downloads, encoder training, cache extraction or Streamlit changes occur here.
Labels: **0 = Fake, 1 = Real**. Validation macro-F1 selects the checkpoint and seed. Test evaluation follows selection. Fargo is qualitative only.
Missing or inconsistent source artifacts stop execution; they are never replaced with synthetic data.


In [ ]:
%pip -q install "transformers==4.57.1" "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" "Pillow>=10,<13" safetensors
import torch, torchvision
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU.'
print(torch.cuda.get_device_name(0), torch.__version__, torchvision.__version__)


## 1. Mount Drive and write isolated embedded source

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, json, hashlib, shutil
drive.mount('/content/drive')
WORK = Path('/content/fakeddit_gated_code')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
SOURCE_FILES = {'dataset.py': '"""Read existing paired data without rewriting CSVs or downloading images."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport pandas as pd\nfrom PIL import Image\n\nSIZES = {"train": 3000, "validate": 500, "test": 500}\nLABELS = {0: "Fake", 1: "Real"}\n\n\ndef resolve_image(root, raw):\n    root = Path(root).resolve()\n    parts = str(raw).replace("\\\\", "/").split("/")\n    # Relocate even absolute Windows paths by the images/ suffix.\n    if "images" not in parts:\n        raise ValueError(f"Image path has no images directory: {raw}")\n    relative = Path(*parts[parts.index("images"):])\n    target = (root / relative).resolve()\n    if not target.is_relative_to(root):\n        raise ValueError(f"Unsafe image path: {raw}")\n    return target\n\n\ndef extract_zip(archive, destination):\n    destination = Path(destination).resolve()\n    destination.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as z:\n        for entry in z.infolist():\n            name = entry.filename.replace("\\\\", "/")\n            target = (destination / name).resolve()\n            if not target.is_relative_to(destination) or ":" in name:\n                raise ValueError(f"Unsafe ZIP member: {name}")\n            if (entry.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError("ZIP symlinks are not supported")\n        for entry in z.infolist():\n            target = destination / entry.filename.replace("\\\\", "/")\n            if entry.is_dir():\n                target.mkdir(parents=True, exist_ok=True)\n            else:\n                target.parent.mkdir(parents=True, exist_ok=True)\n                with z.open(entry) as src, target.open("wb") as dst:\n                    import shutil\n                    shutil.copyfileobj(src, dst)\n    matches = list(destination.rglob("train_processed.csv"))\n    if len(matches) != 1:\n        raise ValueError("ZIP must contain exactly one processed dataset")\n    return matches[0].parent\n\n\ndef validate(root, report_path):\n    frames, report, errors = {}, {"label_mapping": LABELS, "splits": {}}, []\n    for split, expected in SIZES.items():\n        df = pd.read_csv(Path(root) / f"{split}_processed.csv", dtype={"id": str}, keep_default_na=False)\n        required = ["id", "clean_title", "2_way_label", "local_image_path"]\n        if not set(required).issubset(df.columns):\n            raise ValueError(f"Missing columns in {split}: {set(required) - set(df.columns)}")\n        missing = {c: int(df[c].astype(str).str.strip().eq("").sum()) for c in required}\n        duplicates = df.loc[df.id.duplicated(), "id"].tolist()\n        if len(df) != expected or any(missing.values()) or duplicates:\n            errors.append(f"{split}: count/missing/duplicate failure")\n        numeric = pd.to_numeric(df["2_way_label"], errors="coerce")\n        if not numeric.isin([0, 1]).all():\n            errors.append(f"{split}: labels must be 0 or 1")\n        if "6_way_label" in df:\n            six = pd.to_numeric(df["6_way_label"], errors="coerce")\n            if not six.isin(range(6)).all() or not ((six == 0) == (numeric == 1)).all():\n                errors.append(f"{split}: binary labels inconsistent with 6-way true category")\n        paths, hashes, bad = [], [], []\n        for row in df.itertuples(index=False):\n            try:\n                p = resolve_image(root, row.local_image_path)\n                with Image.open(p) as im:\n                    im.verify()\n                with Image.open(p) as im:\n                    im.convert("RGB").load()\n                hashes.append(hashlib.sha256(p.read_bytes()).hexdigest())\n                paths.append(str(p))\n            except Exception as exc:\n                bad.append({"id": row.id, "error": str(exc)})\n                paths.append("")\n                hashes.append("")\n        if bad:\n            errors.append(f"{split}: {len(bad)} missing/corrupt images")\n        df["resolved_path"], df["image_sha256"] = paths, hashes\n        frames[split] = df\n        report["splits"][split] = {"count": len(df), "classes": df["2_way_label"].value_counts().to_dict(),\n                                  "missing": missing, "duplicate_ids": duplicates, "bad_images": bad}\n    names = list(frames)\n    report["overlaps"] = {}\n    for i, a in enumerate(names):\n        for b in names[i + 1:]:\n            overlap = sorted(set(frames[a].id) & set(frames[b].id))\n            report["overlaps"][f"{a}/{b}"] = overlap\n            if overlap:\n                errors.append(f"{a}/{b}: overlapping sample IDs")\n    report["errors"] = errors\n    Path(report_path).parent.mkdir(parents=True, exist_ok=True)\n    Path(report_path).write_text(json.dumps(report, indent=2), encoding="utf-8")\n    if errors:\n        raise ValueError(f"Validation failed; see {report_path}: {errors}")\n    return frames, report\n\n\nif __name__ == "__main__":\n    _, report = validate("data", "results/dataset_validation.json")\n    print(json.dumps(report, indent=2))\n', 'models.py': '"""Shared Colab/Windows model contract. Backbones are always frozen."""\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom PIL import ImageOps\nfrom torchvision.models import resnet50, ResNet50_Weights\nfrom transformers import AutoModel, AutoTokenizer\n\nENCODER_CONFIG = {"text_model": "distilbert-base-uncased", "max_length": 128,\n                  "pooling": "attention_mask_mean", "image_weights": "IMAGENET1K_V2",\n                  "text_dim": 768, "image_dim": 2048, "version": 1}\nHEAD_CONFIG = {"text": {"dimensions": [768, 256, 2], "dropout": 0.3},\n               "image": {"dimensions": [2048, 256, 2], "dropout": 0.3},\n               "fusion": {"text_projection": [768, 256], "image_projection": [2048, 256],\n                          "dimensions": [512, 128, 2], "dropout": 0.3}, "version": 1}\n\n\nclass Head(nn.Module):\n    def __init__(self, kind):\n        super().__init__()\n        if kind not in ("text", "image", "fusion"):\n            raise ValueError(kind)\n        self.kind = kind\n        if kind == "fusion":\n            self.text = nn.Linear(768, 256)\n            self.image = nn.Linear(2048, 256)\n            self.classifier = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(.3), nn.Linear(128, 2))\n        else:\n            self.classifier = nn.Sequential(nn.Linear(768 if kind == "text" else 2048, 256),\n                                            nn.ReLU(), nn.Dropout(.3), nn.Linear(256, 2))\n\n    def forward(self, text, image):\n        x = torch.cat((self.text(text), self.image(image)), dim=1) if self.kind == "fusion" else (text if self.kind == "text" else image)\n        return self.classifier(x)\n\n\ndef masked_mean(hidden, mask):\n    mask = mask.unsqueeze(-1).to(hidden.dtype)\n    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)\n\n\nclass Encoders:\n    def __init__(self, assets, device="cpu", download=False):\n        self.device = torch.device(device)\n        assets = Path(assets)\n        text_dir = assets / "distilbert"\n        source = ENCODER_CONFIG["text_model"] if download else str(text_dir)\n        self.tokenizer = AutoTokenizer.from_pretrained(source, local_files_only=not download)\n        self.text = AutoModel.from_pretrained(source, local_files_only=not download)\n        weights = ResNet50_Weights.IMAGENET1K_V2\n        self.image = resnet50(weights=weights if download else None)\n        if download:\n            assets.mkdir(parents=True, exist_ok=True)\n            self.tokenizer.save_pretrained(text_dir)\n            self.text.save_pretrained(text_dir)\n            torch.save(self.image.state_dict(), assets / "resnet50.pt")\n        else:\n            self.image.load_state_dict(torch.load(assets / "resnet50.pt", map_location="cpu", weights_only=True))\n        self.image.fc = nn.Identity()\n        self.transform = weights.transforms()\n        for model in (self.text, self.image):\n            model.requires_grad_(False).eval().to(self.device)\n\n    @torch.inference_mode()\n    def encode_text(self, titles):\n        tokens = self.tokenizer(titles, padding=True, truncation=True, max_length=128, return_tensors="pt").to(self.device)\n        return masked_mean(self.text(**tokens).last_hidden_state, tokens["attention_mask"]).float()\n\n    @torch.inference_mode()\n    def encode_images(self, images):\n        batch = torch.stack([self.transform(ImageOps.exif_transpose(im).convert("RGB")) for im in images])\n        return self.image(batch.to(self.device)).float()\n', 'pipeline.py': '"""GPU-only bulk extraction/training; persistent, fingerprinted recovery files."""\nimport hashlib\nimport json\nimport os\nimport pickle\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom PIL import Image\nfrom sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix\nfrom models import ENCODER_CONFIG, Head\n\n\ndef atomic_save(obj, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    torch.save(obj, tmp)\n    os.replace(tmp, path)\n\n\ndef write_json(obj, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(".json.tmp")\n    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef fingerprint(frames, config):\n    records = {s: df[["id", "clean_title", "2_way_label", "image_sha256"]].to_dict("records") for s, df in frames.items()}\n    return hashlib.sha256(json.dumps({"data": records, "config": config}, sort_keys=True).encode()).hexdigest()[:20]\n\n\ndef gpu():\n    if not torch.cuda.is_available():\n        raise RuntimeError("Select Colab Runtime > Change runtime type > GPU, then rerun. Bulk CPU work is disabled.")\n    return torch.device("cuda")\n\n\ndef extract(frames, encoders, cache, batch_size=32):\n    gpu()\n    cache = Path(cache)\n    result = {}\n    for split, df in frames.items():\n        # Separate modality chunks let a completed text batch survive an image failure.\n        features = {}\n        for modality, dim in (("text", 768), ("image", 2048)):\n            chunks = []\n            for start in range(0, len(df), batch_size):\n                batch = df.iloc[start:start + batch_size]\n                ids = batch.id.tolist()\n                labels = torch.tensor(batch["2_way_label"].tolist(), dtype=torch.long)\n                path = cache / split / modality / f"{start:06d}-{len(batch):04d}.pt"\n                part = None\n                if path.exists():\n                    try:\n                        saved = torch.load(path, map_location="cpu", weights_only=True)\n                        if (saved["ids"] == ids and torch.equal(saved["labels"], labels)\n                                and saved["features"].shape == (len(batch), dim)\n                                and torch.isfinite(saved["features"]).all()):\n                            part = saved\n                    except (OSError, RuntimeError, EOFError, ValueError, KeyError, pickle.UnpicklingError):\n                        print("Recomputing unreadable chunk:", path)\n                if part is None:\n                    if modality == "text":\n                        embedding = encoders.encode_text(batch.clean_title.tolist())\n                    else:\n                        images = []\n                        for filename in batch.resolved_path:\n                            with Image.open(filename) as im:\n                                images.append(im.convert("RGB"))\n                        embedding = encoders.encode_images(images)\n                    part = {"ids": ids, "labels": labels, "features": embedding.cpu()}\n                    if not torch.isfinite(part["features"]).all():\n                        raise ValueError("Non-finite embeddings")\n                    atomic_save(part, path)\n                chunks.append(part["features"])\n            features[modality] = torch.cat(chunks)\n            print(f"{split}/{modality}: {len(df)} cached features ready")\n        result[split] = {**features, "labels": torch.tensor(df["2_way_label"].tolist(), dtype=torch.long), "ids": df.id.tolist()}\n    return result\n\n\ndef train(kind, features, run, config, run_id):\n    device = gpu()\n    run = Path(run)\n    last_path = run / "recovery" / f"{kind}.pt"\n    best_path = run / "models" / f"{kind}.pt"\n    seed = config["seed"]\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    model = Head(kind).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])\n    loss_fn = nn.CrossEntropyLoss()\n    start, best, stale, history, best_state = 0, float("inf"), 0, [], None\n    if last_path.exists():\n        state = torch.load(last_path, map_location="cpu", weights_only=True)\n        if state["run_id"] != run_id:\n            raise ValueError("Recovery fingerprint mismatch")\n        model.load_state_dict(state["state_dict"])\n        optimizer.load_state_dict(state["optimizer"])\n        for opt in optimizer.state.values():\n            for key, value in opt.items():\n                if isinstance(value, torch.Tensor):\n                    opt[key] = value.to(device)\n        start, best, stale, history = state["epoch"] + 1, state["best"], state["stale"], state["history"]\n        best_state = state["best_state"]\n        torch.set_rng_state(state["rng"])\n        torch.cuda.set_rng_state_all(state["cuda_rng"])\n        print(f"{kind}: recovering after epoch {start}")\n    data = {s: {k: v.to(device) for k, v in features[s].items() if k != "ids"} for s in ("train", "validate")}\n    for epoch in range(start, config["epochs"]):\n        if stale >= config["patience"]:\n            break\n        model.train()\n        order = torch.randperm(len(data["train"]["labels"]), device=device)\n        total = 0.0\n        for idx in order.split(config["batch_size"]):\n            d = data["train"]\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(d["text"][idx], d["image"][idx]), d["labels"][idx])\n            loss.backward()\n            optimizer.step()\n            total += loss.item() * len(idx)\n        model.eval()\n        with torch.inference_mode():\n            d = data["validate"]\n            val = loss_fn(model(d["text"], d["image"]), d["labels"]).item()\n        if not np.isfinite(val):\n            raise ValueError("Non-finite validation loss")\n        history.append({"epoch": epoch + 1, "train_loss": total / len(order), "val_loss": val})\n        if val < best:\n            best, stale = val, 0\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            stale += 1\n        # One recovery file contains both last and best states, avoiding split-file recovery races.\n        atomic_save({"run_id": run_id, "epoch": epoch, "state_dict": model.state_dict(),\n                     "optimizer": optimizer.state_dict(), "best_state": best_state, "best": best,\n                     "stale": stale, "history": history, "rng": torch.get_rng_state(),\n                     "cuda_rng": torch.cuda.get_rng_state_all()}, last_path)\n        print(f"{kind} epoch {epoch + 1}: train={history[-1][\'train_loss\']:.4f} val={val:.4f}")\n    if best_state is None:\n        raise ValueError("No trained checkpoint available; epochs must be positive")\n    atomic_save({"state_dict": best_state, "kind": kind, "run_id": run_id}, best_path)\n    write_json(history, run / "results" / f"{kind}_history.json")\n    return history\n\n\ndef evaluate(features, run, run_id):\n    device = gpu()\n    run = Path(run)\n    test = features["test"]\n    assert len(test["ids"]) == 500, "Use the unchanged 500-example test split"\n    y = test["labels"].numpy()\n    metrics, predictions = {}, {"id": test["ids"], "label": y.tolist()}\n    for kind in ("text", "image", "fusion"):\n        checkpoint = torch.load(run / "models" / f"{kind}.pt", map_location="cpu", weights_only=True)\n        assert checkpoint["run_id"] == run_id\n        model = Head(kind).to(device).eval()\n        model.load_state_dict(checkpoint["state_dict"])\n        with torch.inference_mode():\n            probabilities = model(test["text"].to(device), test["image"].to(device)).softmax(1).cpu().numpy()\n        pred = probabilities.argmax(1)\n        p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", pos_label=0, zero_division=0)\n        metrics[kind] = {"accuracy": float(accuracy_score(y, pred)), "precision": float(p), "recall": float(r),\n                         "f1": float(f), "roc_auc": float(roc_auc_score(y == 0, probabilities[:, 0])),\n                         "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist()}\n        predictions[f"{kind}_prediction"] = pred.tolist()\n        predictions[f"{kind}_fake_probability"] = probabilities[:, 0].tolist()\n    result = {"run_id": run_id, "test_count": 500, "positive_class": "Fake (0)", "models": metrics,\n              "fusion_outperforms_both": {metric: all(metrics["fusion"][metric] > metrics[k][metric] for k in ("text", "image"))\n                                          for metric in ("accuracy", "f1")}}\n    write_json(result, run / "results" / "metrics.json")\n    import pandas as pd\n    pd.DataFrame(predictions).to_csv(run / "results" / "predictions.csv", index=False)\n    import matplotlib\n    matplotlib.use("Agg")  # Export figures without a desktop/Tk dependency.\n    import matplotlib.pyplot as plt\n    pd.DataFrame(metrics).T[["accuracy", "precision", "recall", "f1", "roc_auc"]].plot.bar(ylim=(0, 1), figsize=(10, 5))\n    plt.tight_layout()\n    plt.savefig(run / "results" / "comparison.png")\n    plt.close()\n    for kind in metrics:\n        fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n        matrix = np.array(metrics[kind]["confusion_matrix"])\n        axes[0].imshow(matrix, cmap="Blues")\n        for (i, j), value in np.ndenumerate(matrix):\n            axes[0].text(j, i, str(value), ha="center")\n        axes[0].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"], xlabel="Predicted", ylabel="Actual")\n        history = pd.read_json(run / "results" / f"{kind}_history.json")\n        axes[1].plot(history.epoch, history.train_loss, label="Train")\n        axes[1].plot(history.epoch, history.val_loss, label="Validation")\n        axes[1].legend()\n        axes[1].set(xlabel="Epoch", ylabel="Cross-entropy")\n        fig.suptitle(kind)\n        fig.tight_layout()\n        fig.savefig(run / "results" / f"{kind}.png")\n        plt.close(fig)\n    return result\n', 'clip_models.py': '"""Experimental CLIP heads and offline encoders; independent of the production app."""\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom PIL import ImageOps\nfrom transformers import AutoModel, AutoTokenizer, CLIPImageProcessor, CLIPVisionModelWithProjection\n\nCLIP_ID = "openai/clip-vit-base-patch32"\nFEATURE_SPEC = {"model": CLIP_ID, "output": "image_embeds", "projection_dim": 512,\n                "l2_normalize": True, "processor": "CLIPImageProcessor", "version": 1}\n\n\nclass ClipHead(nn.Module):\n    def __init__(self, kind, text_dim=768, image_dim=512, dropout=0.3):\n        super().__init__()\n        if kind not in ("clip_image", "clip_fusion"):\n            raise ValueError(f"Unknown experimental head: {kind}")\n        if text_dim <= 0 or image_dim <= 0:\n            raise ValueError("Feature dimensions must be positive")\n        self.spec = dict(kind=kind, text_dim=text_dim, image_dim=image_dim, dropout=dropout)\n        self.kind = kind\n        if kind == "clip_fusion":\n            self.text_projection = nn.Linear(text_dim, 256)\n            self.image_projection = nn.Linear(image_dim, 256)\n            self.classifier = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 2))\n        else:\n            self.classifier = nn.Sequential(nn.Linear(image_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 2))\n\n    def forward(self, text, image):\n        if image.ndim != 2 or image.shape[1] != self.spec["image_dim"]:\n            raise ValueError("Wrong CLIP image feature dimension")\n        if self.kind == "clip_fusion":\n            if text.ndim != 2 or text.shape != (len(image), self.spec["text_dim"]):\n                raise ValueError("Wrong text feature dimension or pairing")\n            image = torch.cat([self.text_projection(text), self.image_projection(image)], dim=1)\n        return self.classifier(image)\n\n\nclass ClipEncoder:\n    def __init__(self, source, device="cpu", revision=None, local=True):\n        kwargs = {"local_files_only": local}\n        if revision is not None:\n            kwargs["revision"] = revision\n        self.processor = CLIPImageProcessor.from_pretrained(str(source), **kwargs)\n        self.model = CLIPVisionModelWithProjection.from_pretrained(str(source), **kwargs)\n        self.model.requires_grad_(False).eval().to(device)\n        self.device = torch.device(device)\n        self.dimension = self.model.config.projection_dim\n        if self.dimension != FEATURE_SPEC["projection_dim"]:\n            raise ValueError("Unexpected CLIP projection size")\n\n    def save(self, path):\n        self.model.save_pretrained(path, safe_serialization=True)\n        self.processor.save_pretrained(path)\n\n    @torch.inference_mode()\n    def encode(self, images):\n        images = [ImageOps.exif_transpose(im).convert("RGB") for im in images]\n        inputs = self.processor(images=images, return_tensors="pt")\n        image = self.model(pixel_values=inputs.pixel_values.to(self.device)).image_embeds.float()\n        return F.normalize(image, p=2, dim=-1)\n\n\nclass CachedTextEncoder:\n    """Use exactly the baseline\'s exported DistilBERT and masked-mean preprocessing."""\n    def __init__(self, source, device="cpu"):\n        self.tokenizer = AutoTokenizer.from_pretrained(str(source), local_files_only=True)\n        self.model = AutoModel.from_pretrained(str(source), local_files_only=True)\n        self.model.requires_grad_(False).eval().to(device)\n        self.device = torch.device(device)\n\n    @torch.inference_mode()\n    def encode(self, titles):\n        tokens = self.tokenizer(titles, padding=True, truncation=True, max_length=128, return_tensors="pt").to(self.device)\n        hidden = self.model(**tokens).last_hidden_state\n        mask = tokens["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n        return ((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).float()\n', 'clip_experiment.py': '"""Isolated GPU experiment. Baseline modules, caches, and checkpoints are read-only."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport pickle\nimport shutil\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, roc_auc_score, confusion_matrix\n\nfrom models import Head, ENCODER_CONFIG\nfrom pipeline import atomic_save, write_json, fingerprint, gpu\nfrom clip_models import ClipEncoder, ClipHead, FEATURE_SPEC, CLIP_ID\n\nEXPERIMENT_VERSION = 1\nKINDS = ("clip_image", "clip_fusion")\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()[:20]\n\n\ndef sha_file(path):\n    with Path(path).open("rb") as file:\n        return hashlib.file_digest(file, "sha256").hexdigest()\n\n\ndef guard_output(output, *protected):\n    output = Path(output).resolve()\n    for path in protected:\n        path = Path(path).resolve()\n        if output.is_relative_to(path) or path.is_relative_to(output):\n            raise ValueError(f"Experiment output must be separate from protected path: {path}")\n    return output\n\n\ndef read_baseline_config(run, frames):\n    run = Path(run)\n    config = json.loads((run / "config.json").read_text(encoding="utf-8"))\n    if config["encoders"] != ENCODER_CONFIG or config["labels"] != {"0": "Fake", "1": "Real"}:\n        raise ValueError("Baseline encoder or label contract differs from this experiment")\n    expected = fingerprint(frames, config["encoders"])\n    if config["feature_id"] != expected:\n        raise ValueError("Baseline dataset fingerprint mismatch; do not mix different processed datasets")\n    provenance = {"config": config, "checkpoint_sha256": {}}\n    for kind in ("text", "image", "fusion"):\n        path = run / "models" / f"{kind}.pt"\n        saved = torch.load(path, map_location="cpu", weights_only=True)\n        if saved["run_id"] != config["run_id"] or saved["kind"] != kind:\n            raise ValueError("Baseline checkpoints do not belong to the selected run")\n        provenance["checkpoint_sha256"][kind] = sha_file(path)\n    return config, provenance\n\n\ndef read_cached_modality(folder, frame, dimension):\n    """Join by ID and validate labels; never trust chunk order or row positions."""\n    rows = {}\n    paths = sorted(Path(folder).glob("*.pt"))\n    if not paths:\n        raise FileNotFoundError(f"No existing features in {folder}. Restore the original Drive feature cache; this experiment does not regenerate it.")\n    for path in paths:\n        part = torch.load(path, map_location="cpu", weights_only=True)\n        ids, labels, features = part["ids"], part["labels"], part["features"]\n        if (features.shape != (len(ids), dimension) or labels.shape != (len(ids),)\n                or labels.dtype != torch.long or not torch.isfinite(features).all()):\n            raise ValueError(f"Invalid cached feature dimensions/values: {path}")\n        for index, sample_id in enumerate(ids):\n            label, embedding = int(labels[index]), features[index].float()\n            if sample_id in rows:\n                old_label, old_embedding = rows[sample_id]\n                if label != old_label or not torch.allclose(embedding, old_embedding, atol=1e-5, rtol=1e-5):\n                    raise ValueError(f"Conflicting duplicate cache ID: {sample_id}")\n            rows[sample_id] = (label, embedding)\n    if set(rows) != set(frame.id) or len(frame.id) != len(set(frame.id)):\n        raise ValueError(f"Missing/extra IDs in {folder}; cannot preserve split alignment")\n    expected = frame["2_way_label"].astype(int).tolist()\n    if [rows[sample_id][0] for sample_id in frame.id] != expected:\n        raise ValueError(f"Cached labels disagree with CSV in {folder}")\n    return torch.stack([rows[sample_id][1] for sample_id in frame.id])\n\n\ndef load_baseline_features(cache_root, feature_id, frames):\n    result = {}\n    for split, frame in frames.items():\n        folder = Path(cache_root) / feature_id / split\n        result[split] = {\n            "ids": frame.id.tolist(), "labels": torch.tensor(frame["2_way_label"].tolist(), dtype=torch.long),\n            "text": read_cached_modality(folder / "text", frame, 768),\n            "image": read_cached_modality(folder / "image", frame, 2048),\n        }\n    return result\n\n\ndef prepare_clip(output, revision="main"):\n    """Resolve once to an immutable HF revision; future recovery is offline."""\n    gpu()\n    output = Path(output)\n    request = {"model": CLIP_ID, "requested_revision": revision}\n    root = output / "encoders" / digest(request)\n    marker = root / "READY.json"\n    if marker.exists():\n        manifest = json.loads(marker.read_text())\n        if manifest["request"] != request or manifest["feature_spec"] != FEATURE_SPEC:\n            raise ValueError("CLIP asset manifest mismatch")\n        encoder = ClipEncoder(root, "cuda", local=True)\n    else:\n        from huggingface_hub import model_info\n        commit = model_info(CLIP_ID, revision=revision).sha\n        encoder = ClipEncoder(CLIP_ID, "cuda", revision=commit, local=False)\n        root.mkdir(parents=True, exist_ok=True)\n        encoder.save(root)\n        manifest = {"request": request, "resolved_revision": commit, "feature_spec": FEATURE_SPEC,\n                    "sha256": {p.name: sha_file(p) for p in root.iterdir()\n                               if p.is_file() and not p.name.endswith(".tmp")}}\n        write_json(manifest, marker)\n    for name, expected in manifest["sha256"].items():\n        if sha_file(root / name) != expected:\n            raise ValueError(f"CLIP asset changed: {name}; restore it or use a new experiment output directory")\n    return encoder, root, manifest\n\n\ndef extract_clip(frames, encoder, folder, feature_id, batch_size=32):\n    gpu()\n    if batch_size < 1:\n        raise ValueError("Feature batch size must be positive")\n    folder = Path(folder) / feature_id\n    result = {}\n    for split, frame in frames.items():\n        # Fixed shards are independent of GPU batch size: lowering the batch size retains completed work.\n        chunks = []\n        for start in range(0, len(frame), 128):\n            batch = frame.iloc[start:start + 128]\n            labels = torch.tensor(batch["2_way_label"].tolist(), dtype=torch.long)\n            ids = batch.id.tolist()\n            path = folder / split / f"{start:06d}.pt"\n            saved = None\n            if path.exists():\n                try:\n                    saved = torch.load(path, map_location="cpu", weights_only=True)\n                    if (saved["feature_id"] != feature_id or saved["ids"] != ids or not torch.equal(saved["labels"], labels)\n                            or saved["features"].shape != (len(ids), encoder.dimension)\n                            or not torch.isfinite(saved["features"]).all()):\n                        saved = None\n                except (OSError, RuntimeError, EOFError, ValueError, KeyError, pickle.UnpicklingError):\n                    saved = None\n            if saved is None:\n                embeddings = []\n                for offset in range(0, len(batch), batch_size):\n                    images = []\n                    for filename in batch.iloc[offset:offset + batch_size].resolved_path:\n                        with Image.open(filename) as image:\n                            images.append(image.copy())\n                    embeddings.append(encoder.encode(images).cpu())\n                feature = torch.cat(embeddings)\n                if feature.shape != (len(ids), encoder.dimension) or not torch.isfinite(feature).all():\n                    raise ValueError("Invalid CLIP embeddings")\n                saved = {"feature_id": feature_id, "ids": ids, "labels": labels, "features": feature}\n                atomic_save(saved, path)\n            chunks.append(saved["features"])\n        result[split] = torch.cat(chunks)\n        print(f"CLIP {split}: {len(frame)} features ready")\n    return result\n\n\ndef paired_features(baseline, visual):\n    return {split: {**data, "image": visual[split]} for split, data in baseline.items()}\n\n\ndef cache_hashes(features):\n    return {split: {modality: hashlib.sha256(data[modality].contiguous().numpy().tobytes()).hexdigest()\n                    for modality in ("text", "image")} for split, data in features.items()}\n\n\ndef binary_metrics(labels, probabilities):\n    y = np.asarray(labels)\n    probabilities = np.asarray(probabilities)\n    if probabilities.shape != (len(y), 2) or not np.isfinite(probabilities).all():\n        raise ValueError("Invalid binary probabilities")\n    pred = probabilities.argmax(1)\n    precision, recall, f1, _ = precision_recall_fscore_support(y, pred, average="binary", pos_label=0, zero_division=0)\n    return {"accuracy": float(accuracy_score(y, pred)), "precision": float(precision), "recall": float(recall),\n            "f1": float(f1), "macro_f1": float(f1_score(y, pred, labels=[0, 1], average="macro", zero_division=0)),\n            "roc_auc": float(roc_auc_score(y == 0, probabilities[:, 0])) if len(set(y)) == 2 else None,\n            "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist()}\n\n\n@torch.inference_mode()\ndef probabilities(model, data, device):\n    model.to(device).eval()\n    output = []\n    for start in range(0, len(data["labels"]), 128):\n        logits = model(data["text"][start:start + 128].to(device), data["image"][start:start + 128].to(device))\n        output.append(logits.softmax(1).cpu())\n    return torch.cat(output).numpy()\n\n\ndef train_trial(kind, data, root, config, run_id):\n    """Validation macro-F1 selects epochs; validation loss breaks exact ties."""\n    device = gpu()\n    spec = dict(kind=kind, text_dim=data["train"]["text"].shape[1], image_dim=data["train"]["image"].shape[1], dropout=config["dropout"])\n    trial_id = digest({"run_id": run_id, "spec": spec, "training": config})\n    root = Path(root) / "trials" / kind / trial_id\n    last = root / "recovery.pt"\n    root.mkdir(parents=True, exist_ok=True)\n    if config["epochs"] < 1 or config["patience"] < 1 or config["batch_size"] < 1:\n        raise ValueError("Epochs, patience, and batch size must be positive")\n    torch.manual_seed(config["seed"])\n    torch.cuda.manual_seed_all(config["seed"])\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    model = ClipHead(**spec).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])\n    start, stale, best_score, best_loss, best_state, history = 0, 0, -1., float("inf"), None, []\n    if last.exists():\n        state = torch.load(last, map_location="cpu", weights_only=True)\n        if state["trial_id"] != trial_id or state["run_id"] != run_id:\n            raise ValueError("Trial recovery mismatch")\n        model.load_state_dict(state["state_dict"])\n        optimizer.load_state_dict(state["optimizer"])\n        # AdamW maps moment buffers to the parameter device during load_state_dict.\n        start, stale = state["epoch"] + 1, state["stale"]\n        best_score, best_loss, best_state, history = state["best_score"], state["best_loss"], state["best_state"], state["history"]\n        torch.set_rng_state(state["rng"])\n        torch.cuda.set_rng_state_all(state["cuda_rng"])\n    # Test tensors are never accessed by training or trial selection.\n    train = {k: data["train"][k].to(device) for k in ("text", "image", "labels")}\n    val = {k: data["validate"][k].to(device) for k in ("text", "image", "labels")}\n    loss_fn = nn.CrossEntropyLoss()\n    for epoch in range(start, config["epochs"]):\n        if stale >= config["patience"]:\n            break\n        model.train()\n        order = torch.randperm(len(train["labels"]), device=device)\n        total = 0.\n        for ids in order.split(config["batch_size"]):\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(train["text"][ids], train["image"][ids]), train["labels"][ids])\n            loss.backward()\n            optimizer.step()\n            total += loss.item() * len(ids)\n        model.eval()\n        with torch.inference_mode():\n            logits = model(val["text"], val["image"])\n            val_loss = loss_fn(logits, val["labels"]).item()\n            metric = binary_metrics(val["labels"].cpu().numpy(), logits.softmax(1).cpu().numpy())\n        if not np.isfinite(val_loss):\n            raise ValueError("Non-finite validation loss")\n        score = metric["macro_f1"]\n        if (score, -val_loss) > (best_score, -best_loss):\n            best_score, best_loss, stale = score, val_loss, 0\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            stale += 1\n        history.append({"epoch": epoch + 1, "train_loss": total / len(order), "validation_loss": val_loss, "validation_macro_f1": score})\n        atomic_save({"run_id": run_id, "trial_id": trial_id, "epoch": epoch, "stale": stale,\n                     "best_score": best_score, "best_loss": best_loss, "best_state": best_state, "history": history,\n                     "state_dict": model.state_dict(), "optimizer": optimizer.state_dict(),\n                     "rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all()}, last)\n        print(f"{kind} {trial_id} epoch {epoch + 1}: validation macro-F1={score:.4f}")\n    if best_state is None:\n        raise ValueError("No completed training epoch")\n    checkpoint = {"run_id": run_id, "trial_id": trial_id, "spec": spec, "state_dict": best_state,\n                  "training": config, "validation_macro_f1": best_score, "validation_loss": best_loss}\n    atomic_save(checkpoint, root / "best.pt")\n    write_json(history, root / "history.json")\n    return root / "best.pt"\n\n\ndef select_trials(data, run, configs, run_id):\n    run = Path(run)\n    selected = {}\n    for kind in KINDS:\n        candidates = [train_trial(kind, data, run, cfg, run_id) for cfg in configs]\n        records = [torch.load(path, map_location="cpu", weights_only=True) for path in candidates]\n        index = max(range(len(records)), key=lambda i: (records[i]["validation_macro_f1"], -records[i]["validation_loss"]))\n        saved = records[index]\n        atomic_save(saved, run / "models" / f"{kind}.pt")\n        selected[kind] = {k: saved[k] for k in ("trial_id", "spec", "training", "validation_macro_f1", "validation_loss")}\n        selected[kind]["all_trials"] = [{k: r[k] for k in ("trial_id", "training", "validation_macro_f1", "validation_loss")} for r in records]\n    write_json({"selection_metric": "validation macro-F1; ties: lower validation loss, then grid order", "selected": selected}, run / "selection.json")\n    return selected\n\n\ndef compare_models(baseline, clip_data, baseline_run, run, run_id, splits=("validate",)):\n    device = gpu()\n    run, baseline_run = Path(run), Path(baseline_run)\n    (run / "results").mkdir(parents=True, exist_ok=True)\n    results = {}\n    base_config = json.loads((baseline_run / "config.json").read_text())\n    for split in splits:\n        if split not in ("validate", "test") or len(baseline[split]["ids"]) != 500:\n            raise ValueError("Comparison requires the unchanged 500-example validation/test split")\n        if baseline[split]["ids"] != clip_data[split]["ids"] or not torch.equal(baseline[split]["labels"], clip_data[split]["labels"]):\n            raise ValueError("Comparison split pairing mismatch")\n        rows, metrics = {}, {}\n        for name, old_kind in (("distilbert", "text"), ("resnet50", "image"), ("distilbert_resnet50", "fusion"), ("clip_image", None), ("clip_fusion", None)):\n            if old_kind:\n                saved = torch.load(baseline_run / "models" / f"{old_kind}.pt", map_location="cpu", weights_only=True)\n                if saved["run_id"] != base_config["run_id"] or saved["kind"] != old_kind:\n                    raise ValueError("Invalid baseline checkpoint")\n                model, data = Head(old_kind), baseline[split]\n            else:\n                saved = torch.load(run / "models" / f"{name}.pt", map_location="cpu", weights_only=True)\n                if saved["run_id"] != run_id or saved["spec"]["kind"] != name:\n                    raise ValueError("Invalid CLIP checkpoint")\n                model, data = ClipHead(**saved["spec"]), clip_data[split]\n            model.load_state_dict(saved["state_dict"])\n            probs = probabilities(model, data, device)\n            metrics[name] = binary_metrics(data["labels"].numpy(), probs)\n            rows[f"{name}_prediction"] = probs.argmax(1)\n            rows[f"{name}_fake_probability"] = probs[:, 0]\n            del model\n        pd.DataFrame({"id": baseline[split]["ids"], "label": baseline[split]["labels"].tolist(), **rows}).to_csv(run / "results" / f"{split}_predictions.csv", index=False)\n        result = {"run_id": run_id, "split": split, "count": 500, "positive_class": "Fake (0)",\n                  "confusion_order": ["Fake", "Real"], "models": metrics}\n        write_json(result, run / "results" / f"{split}_metrics.json")\n        pd.DataFrame(metrics).T.drop(columns="confusion_matrix").to_csv(run / "results" / f"{split}_comparison.csv")\n        results[split] = result\n    return results\n\n\ndef promotion_gate(validation, minimum_gain=0.0):\n    if validation["split"] != "validate" or minimum_gain < 0:\n        raise ValueError("Promotion uses validation only and nonnegative minimum gain")\n    metrics = validation["models"]\n    comparisons = {"clip_image": "resnet50", "clip_fusion": "distilbert_resnet50"}\n    return {"criterion": "strict validation macro-F1 improvement", "minimum_gain": minimum_gain,\n            "candidates": {kind: {"baseline": old, "delta_macro_f1": metrics[kind]["macro_f1"] - metrics[old]["macro_f1"],\n                                   "eligible": metrics[kind]["macro_f1"] - metrics[old]["macro_f1"] > minimum_gain}\n                           for kind, old in comparisons.items()},\n            "note": "Eligibility is an observed validation improvement, not statistical significance. Test metrics do not change this decision."}\n\n\ndef plot_comparison(run, split):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    run = Path(run)\n    result = json.loads((run / "results" / f"{split}_metrics.json").read_text())\n    frame = pd.DataFrame(result["models"]).T.drop(columns="confusion_matrix").astype(float)\n    frame.plot.bar(ylim=(0, 1), figsize=(12, 5))\n    plt.tight_layout()\n    plt.savefig(run / "results" / f"{split}_comparison.png")\n    plt.close()\n    fig, axes = plt.subplots(1, len(result["models"]), figsize=(17, 3.5))\n    for ax, (name, metric) in zip(axes, result["models"].items()):\n        matrix = np.array(metric["confusion_matrix"])\n        ax.imshow(matrix, cmap="Blues")\n        for (i, j), count in np.ndenumerate(matrix):\n            ax.text(j, i, str(count), ha="center")\n        ax.set(title=name, xlabel="Predicted", ylabel="Actual", xticks=[0, 1], yticks=[0, 1], xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])\n    fig.tight_layout()\n    fig.savefig(run / "results" / f"{split}_confusion_matrices.png")\n    plt.close(fig)\n\n\ndef export_bundle(run, clip_assets, text_assets, destination, config, source_dir):\n    """Separate clip_artifacts package; no production files or baseline checkpoint copies."""\n    run, destination = Path(run), Path(destination)\n    for name in ("validate_metrics.json", "test_metrics.json", "promotion.json"):\n        if not (run / "results" / name).exists():\n            raise FileNotFoundError(f"Complete evaluation before export: {name}")\n    bundle = destination / "clip_artifacts"\n    bundle.mkdir(parents=True, exist_ok=True)\n    for name in ("models", "results"):\n        shutil.copytree(run / name, bundle / name, dirs_exist_ok=True)\n    shutil.copytree(clip_assets, bundle / "encoders/clip", dirs_exist_ok=True)\n    shutil.copytree(text_assets, bundle / "encoders/distilbert", dirs_exist_ok=True)\n    write_json(config, bundle / "config.json")\n    shutil.copy2(run / "selection.json", bundle / "selection.json")\n    for name in ("clip_models.py", "clip_inference.py", "clip_requirements.txt", "CLIP_EXPERIMENT.md"):\n        shutil.copy2(Path(source_dir) / name, destination / name)\n    return bundle\n', 'gated_model.py': '"""Differentiable gated fusion over existing frozen-encoder features."""\nimport torch\nfrom torch import nn\n\n\nclass GatedFusion(nn.Module):\n    def __init__(self, text_dim, image_dim, branch_dropout=0.2, head_dropout=0.3):\n        super().__init__()\n        if text_dim < 1 or image_dim < 1:\n            raise ValueError("Use positive dimensions measured from verified cached tensors")\n        self.spec = dict(text_dim=text_dim, image_dim=image_dim,\n                         branch_dropout=branch_dropout, head_dropout=head_dropout)\n        def branch(dimension):\n            return nn.Sequential(nn.Linear(dimension, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(branch_dropout))\n        self.text_branch = branch(text_dim)\n        self.image_branch = branch(image_dim)\n        self.gate = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, 2), nn.Softmax(dim=1))\n        self.classifier = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(head_dropout), nn.Linear(128, 2))\n\n    def forward(self, text, image):\n        if text.ndim != 2 or image.ndim != 2 or text.shape != (len(image), self.spec[\'text_dim\']) or image.shape[1] != self.spec[\'image_dim\']:\n            raise ValueError("Wrong feature dimensions or unpaired batches")\n        text, image = self.text_branch(text), self.image_branch(image)\n        gates = self.gate(torch.cat([text, image], dim=1))\n        fused = torch.cat([gates[:, :1] * text, gates[:, 1:] * image], dim=1)\n        return self.classifier(fused), gates[:, 0], gates[:, 1]\n', 'gated_analysis.py': '"""Measured sensitivity and disagreement diagnostics; never relabel samples."""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import (accuracy_score, precision_recall_fscore_support,\n                             roc_auc_score, confusion_matrix, precision_recall_curve)\n\n\ndef save_json(value, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(\'.json.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef metrics(y, fake):\n    y, fake = np.asarray(y), np.asarray(fake)\n    if len(y) == 0 or fake.shape != y.shape or not np.isfinite(fake).all() or not np.isin(y, [0, 1]).all():\n        raise ValueError(\'Expected aligned finite binary predictions\')\n    if np.any((fake < 0) | (fake > 1)):\n        raise ValueError(\'Probabilities must be in [0, 1]\')\n    pred = np.where(fake >= .5, 0, 1)\n    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)\n    return {\'accuracy\': float(accuracy_score(y, pred)), \'macro_f1\': float(f.mean()),\n            \'fake_precision\': float(p[0]), \'fake_recall\': float(r[0]), \'fake_f1\': float(f[0]),\n            \'real_precision\': float(p[1]), \'real_recall\': float(r[1]), \'real_f1\': float(f[1]),\n            \'roc_auc\': float(roc_auc_score(y == 0, fake)) if len(np.unique(y)) == 2 else None,\n            \'confusion_matrix\': confusion_matrix(y, pred, labels=[0, 1]).tolist()}\n\n\n@torch.inference_mode()\ndef forward_batches(model, data, device=\'cpu\', batch_size=128):\n    model.to(device).eval()\n    probabilities, gates = [], []\n    for start in range(0, len(data[\'text\']), batch_size):\n        out = model(data[\'text\'][start:start + batch_size].to(device), data[\'image\'][start:start + batch_size].to(device))\n        if isinstance(out, tuple):\n            logits, text_gate, image_gate = out\n            gates.append(torch.stack([text_gate, image_gate], dim=1).cpu())\n        else:\n            logits = out\n        probabilities.append(logits.softmax(1)[:, 0].cpu())\n    return torch.cat(probabilities).numpy(), torch.cat(gates).numpy() if gates else None\n\n\ndef derangement(count, rng):\n    if count < 2:\n        raise ValueError(\'Swapping requires at least two distinct samples\')\n    # Random cycle: every row receives a different row\'s feature, with no label use.\n    order = rng.permutation(count)\n    mapping = np.empty(count, dtype=int)\n    mapping[order] = np.roll(order, 1)\n    return mapping\n\n\ndef ablations(model, data, training, output, device=\'cpu\', seed=42, repetitions=5, noise_scale=.1):\n    """Only the unperturbed pair retains a meaningful ground-truth evaluation."""\n    if repetitions < 1 or noise_scale < 0:\n        raise ValueError(\'Invalid diagnostic settings\')\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    normal, _ = forward_batches(model, data, device)\n    y = data[\'labels\'].numpy()\n    ids, rows = data[\'ids\'], []\n    reference = {k: training[k].mean(0, keepdim=True).expand_as(data[k]) for k in (\'text\', \'image\')}\n    rng = np.random.default_rng(seed)\n\n    def record(name, text, image, repetition=0, donor=None):\n        changed, _ = forward_batches(model, {\'text\': text, \'image\': image}, device)\n        for i, sample_id in enumerate(ids):\n            rows.append({\'id\': sample_id, \'experiment\': name, \'repetition\': repetition,\n                         \'donor_id\': ids[donor[i]] if donor is not None else None,\n                         \'normal_fake\': float(normal[i]), \'perturbed_fake\': float(changed[i]),\n                         \'delta_fake\': float(changed[i] - normal[i]), \'abs_delta_fake\': float(abs(changed[i] - normal[i])),\n                         \'class_changed\': bool((changed[i] >= .5) != (normal[i] >= .5))})\n\n    for repeat in range(repetitions):\n        # Identical donor mapping for text/image swaps enables paired sensitivity comparison.\n        donor = derangement(len(ids), rng)\n        record(\'swap_image\', data[\'text\'], data[\'image\'][donor], repeat, donor)\n        record(\'swap_text\', data[\'text\'][donor], data[\'image\'], repeat, donor)\n        for key in (\'text\', \'image\'):\n            noise = torch.tensor(rng.standard_normal(tuple(data[key].shape)), dtype=torch.float32)\n            perturbed = data[key] + noise_scale * training[key].std(0, unbiased=False) * noise\n            record(\'noise_\' + key, perturbed if key == \'text\' else data[\'text\'],\n                   perturbed if key == \'image\' else data[\'image\'], repeat)\n    record(\'mean_image\', data[\'text\'], reference[\'image\'])\n    record(\'mean_text\', reference[\'text\'], data[\'image\'])\n    record(\'mean_both\', reference[\'text\'], reference[\'image\'])\n    frame = pd.DataFrame(rows)\n    frame.to_csv(output / \'per_sample_interventions.csv\', index=False)\n    summary = frame.groupby(\'experiment\').agg(mean_abs_probability_change=(\'abs_delta_fake\', \'mean\'),\n        mean_signed_probability_change=(\'delta_fake\', \'mean\'), class_change_rate=(\'class_changed\', \'mean\')).reset_index()\n    summary.to_csv(output / \'summary.csv\', index=False)\n    paired = frame[frame.experiment.isin([\'swap_text\', \'swap_image\'])].pivot_table(\n        index=\'id\', columns=\'experiment\', values=\'abs_delta_fake\', aggfunc=\'mean\')\n    difference = (paired[\'swap_text\'] - paired[\'swap_image\']).to_numpy()\n    boot = np.array([rng.choice(difference, len(difference), replace=True).mean() for _ in range(1000)])\n    low, high = np.quantile(boot, [.025, .975])\n    report = {\'normal_metrics\': metrics(y, normal), \'examples\': len(ids), \'seed\': seed,\n              \'repetitions\': repetitions, \'noise_scale\': noise_scale, \'interventions\': summary.to_dict(\'records\'),\n              \'swap_text_minus_image_abs_change\': float(difference.mean()),\n              \'descriptive_bootstrap_95_interval\': [float(low), float(high)],\n              \'interpretation\': (\'Text swaps were more disruptive in this diagnostic.\' if low > 0 else\n                                 \'Image swaps were more disruptive in this diagnostic.\' if high < 0 else\n                                 \'This diagnostic does not clearly separate text and image swap sensitivity.\'),\n              \'limits\': \'Swaps, Gaussian noise and training means may be out of distribution. No intervention accuracy is claimed because mismatched pairs lack justified labels. Sensitivity and gates are not causal feature importance.\'}\n    save_json(report, output / \'report.json\')\n    plt = plotting()\n    summary.set_index(\'experiment\')[[\'mean_abs_probability_change\', \'class_change_rate\']].plot.bar(figsize=(11, 4))\n    plt.tight_layout(); plt.savefig(output / \'ablation.png\'); plt.close()\n    return report\n\n\ndef disagreement_score(text_fake, image_fake):\n    text, image = np.asarray(text_fake, dtype=float), np.asarray(image_fake, dtype=float)\n    if text.shape != image.shape or not np.isfinite(text).all() or not np.isfinite(image).all() or np.any((text < 0) | (text > 1) | (image < 0) | (image > 1)):\n        raise ValueError(\'Expected aligned probabilities in [0, 1]\')\n    return np.abs(text - image)\n\n\ndef choose_threshold(text, image, split=\'validate\', quantile=.8, fixed=None):\n    if split != \'validate\' or not 0 < quantile < 1:\n        raise ValueError(\'Thresholds must be chosen from validation only\')\n    score = disagreement_score(text, image)\n    threshold = float(np.quantile(score, quantile)) if fixed is None else float(fixed)\n    if not 0 <= threshold <= 1:\n        raise ValueError(\'Threshold must be in [0, 1]\')\n    return {\'threshold\': threshold, \'source_split\': \'validate\', \'method\': \'validation quantile\' if fixed is None else \'predeclared user threshold\',\n            \'quantile\': quantile if fixed is None else None, \'validated_threshold\': False}\n\n\ndef disagreement_report(y, text, image, fused, threshold, split=\'validate\'):\n    score = disagreement_score(text, image)\n    y, fused = np.asarray(y), np.asarray(fused)\n    metrics(y, fused)  # Validate aligned labels and fused probabilities too.\n    if y.shape != score.shape:\n        raise ValueError(\'Disagreement rows are not aligned\')\n    errors = (np.where(fused >= .5, 0, 1) != y)\n    opposite = (np.asarray(text) >= .5) != (np.asarray(image) >= .5)\n    high = score >= threshold[\'threshold\']\n    def group(mask):\n        count = int(mask.sum())\n        return {\'count\': count, \'error_rate\': float(errors[mask].mean()) if count else None}\n    report = {\'split\': split, \'examples\': len(y), \'threshold_policy\': threshold, \'class_disagreement_frequency\': float(opposite.mean()),\n              \'high_score_frequency\': float(high.mean()), \'class_agree\': group(~opposite), \'class_disagree\': group(opposite),\n              \'low_score\': group(~high), \'high_score\': group(high), \'overall_error_rate\': float(errors.mean()),\n              \'error_detection_roc_auc\': float(roc_auc_score(errors, score)) if len(np.unique(errors)) == 2 else None,\n              \'score_error_correlation\': float(np.corrcoef(score, errors.astype(float))[0, 1]) if score.std() > 0 and errors.std() > 0 else None,\n              \'bins\': [{\'lower\': float(a), \'upper\': float(b), **group((score >= a) & ((score < b) if b < 1 else (score <= b)))}\n                       for a, b in zip(np.linspace(0, 1, 6)[:-1], np.linspace(0, 1, 6)[1:])]}\n    return report\n\n\ndef warning_policy(validation_report):\n    if validation_report[\'split\'] != \'validate\' or validation_report[\'threshold_policy\'][\'source_split\'] != \'validate\':\n        raise ValueError(\'Warning policy must use validation data\')\n    high, low = validation_report[\'high_score\'], validation_report[\'low_score\']\n    auc = validation_report[\'error_detection_roc_auc\']\n    enabled = (high[\'count\'] >= 20 and low[\'count\'] >= 20 and high[\'error_rate\'] > low[\'error_rate\']\n               and auc is not None and auc > .5)\n    return {\'enabled\': bool(enabled), \'threshold\': validation_report[\'threshold_policy\'][\'threshold\'],\n            \'criterion\': \'Exploratory validation association: at least 20 cases per group, higher high-score error rate, error-detection AUC > 0.5; not statistical validation.\',\n            \'message\': \'Text and image models disagree substantially. The combined prediction may be less reliable.\',\n            \'changes_label\': False}\n\n\ndef gate_statistics(gates, y=None):\n    gates = np.asarray(gates)\n    if gates.ndim != 2 or gates.shape[1] != 2 or not np.isfinite(gates).all() or np.any((gates < 0) | (gates > 1)) or not np.allclose(gates.sum(1), 1, atol=1e-6):\n        raise ValueError(\'Invalid modality gates\')\n    return {name: {\'mean\': float(gates[:, i].mean()), \'std\': float(gates[:, i].std()),\n                   \'quantiles\': np.quantile(gates[:, i], [0, .25, .5, .75, 1]).tolist()}\n            for i, name in enumerate([\'text\', \'image\'])}\n\n\ndef plotting():\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    return plt\n\n\ndef diagnostic_plots(y, probabilities, gates, history, disagreement, output):\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    plt = plotting()\n    fig, axes = plt.subplots(1, len(probabilities), figsize=(5 * len(probabilities), 4))\n    for ax, (name, fake) in zip(np.atleast_1d(axes), probabilities.items()):\n        matrix = np.array(metrics(y, fake)[\'confusion_matrix\'])\n        ax.imshow(matrix, cmap=\'Blues\')\n        for (i, j), value in np.ndenumerate(matrix):\n            ax.text(j, i, str(value), ha=\'center\')\n        ax.set(title=name, xticks=[0, 1], yticks=[0, 1], xticklabels=[\'Fake\', \'Real\'], yticklabels=[\'Fake\', \'Real\'], xlabel=\'Predicted\', ylabel=\'Actual\')\n    fig.tight_layout(); fig.savefig(output / \'confusion_matrices.png\'); plt.close(fig)\n    for name, fake in probabilities.items():\n        precision, recall, _ = precision_recall_curve(np.asarray(y) == 0, fake)\n        plt.plot(recall, precision, label=name)\n    plt.xlabel(\'Fake recall\'); plt.ylabel(\'Fake precision\'); plt.legend(); plt.tight_layout(); plt.savefig(output / \'precision_recall.png\'); plt.close()\n    plt.hist(gates[:, 0], bins=20, alpha=.6, label=\'Text\'); plt.hist(gates[:, 1], bins=20, alpha=.6, label=\'Image\')\n    plt.legend(); plt.xlabel(\'Gate coefficient (not feature importance)\'); plt.tight_layout(); plt.savefig(output / \'gate_distribution.png\'); plt.close()\n    for key in [\'train_loss\', \'validation_loss\']:\n        plt.plot([r[\'epoch\'] for r in history], [r[key] for r in history], label=key)\n    plt.legend(); plt.xlabel(\'Epoch\'); plt.ylabel(\'Cross-entropy\'); plt.tight_layout(); plt.savefig(output / \'loss_curves.png\'); plt.close()\n    bins = disagreement[\'bins\']\n    plt.bar([f"{b[\'lower\']:.1f}–{b[\'upper\']:.1f}" for b in bins], [b[\'error_rate\'] if b[\'error_rate\'] is not None else 0 for b in bins])\n    plt.xlabel(\'Disagreement score bin (empty bins show 0)\'); plt.ylabel(\'Fusion error rate\'); plt.tight_layout(); plt.savefig(output / \'disagreement_errors.png\'); plt.close()\n', 'gated_pipeline.py': '"""Read-only cache loading, recoverable GPU head training and held-out evaluation."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\n\nfrom pipeline import atomic_save, gpu, fingerprint\nfrom clip_experiment import read_cached_modality, digest, sha_file\nfrom clip_models import ClipHead, FEATURE_SPEC\nfrom models import Head\nfrom gated_model import GatedFusion\nfrom gated_analysis import (save_json, metrics, forward_batches, gate_statistics, disagreement_report,\n                            warning_policy, diagnostic_plots, ablations)\n\nFARGO_TITLE = \'A Fargo, North Dakota, man was arrested for clearing snow with a flamethrower.\'\n\n\ndef load_caches(frames, baseline_bundle, clip_bundle, text_cache_root, clip_cache_root, fargo_image=None):\n    base = json.loads((Path(baseline_bundle) / \'config.json\').read_text())\n    clip = json.loads((Path(clip_bundle) / \'config.json\').read_text())\n    if clip[\'baseline_run_id\'] != base[\'run_id\'] or base[\'labels\'] != {\'0\': \'Fake\', \'1\': \'Real\'} or clip[\'labels\'] != base[\'labels\']:\n        raise ValueError(\'Baseline/CLIP runs or label mappings disagree\')\n    if clip[\'feature_spec\'] != FEATURE_SPEC or fingerprint(frames, base[\'encoders\']) != base[\'feature_id\']:\n        raise ValueError(\'Data/encoder fingerprint differs from the original experiment\')\n    folders = {(split, kind): Path(root) / feature_id / split / (\'text\' if kind == \'text\' else \'\')\n               for split in frames for kind, root, feature_id in [(\'text\', text_cache_root, base[\'feature_id\']), (\'image\', clip_cache_root, clip[\'clip_feature_id\'])]}\n    missing = [str(path / \'*.pt\') for path in folders.values() if not list(path.glob(\'*.pt\'))]\n    if missing:\n        raise FileNotFoundError(\'Required original cache shards are missing; no extraction was attempted:\\n\' + \'\\n\'.join(missing))\n    data, dims = {}, {}\n    for split, frame in frames.items():\n        data[split] = {\'ids\': frame.id.tolist(), \'labels\': torch.tensor(frame[\'2_way_label\'].tolist(), dtype=torch.long)}\n        for kind in (\'text\', \'image\'):\n            folder = folders[split, kind]\n            sample = torch.load(next(iter(sorted(folder.glob(\'*.pt\')))), map_location=\'cpu\', weights_only=True)\n            dimension = sample[\'features\'].shape[1]\n            if kind in dims and dims[kind] != dimension:\n                raise ValueError(\'Dimensions differ between splits\')\n            dims[kind] = dimension\n            if kind == \'image\':\n                for path in folder.glob(\'*.pt\'):\n                    if torch.load(path, map_location=\'cpu\', weights_only=True)[\'feature_id\'] != clip[\'clip_feature_id\']:\n                        raise ValueError(f\'Wrong CLIP shard fingerprint: {path}\')\n            tensor = read_cached_modality(folder, frame, dimension)\n            data[split][kind] = tensor\n            if kind == \'text\':\n                expected = clip[\'provenance\'][\'baseline\'][\'cache_sha256\'][split][\'text\']\n                if hashlib.sha256(tensor.contiguous().numpy().tobytes()).hexdigest() != expected:\n                    raise ValueError(f\'{split}: cached text differs from CLIP training provenance\')\n    if dims != {\'text\': clip[\'heads\'][\'clip_fusion\'][\'text_dim\'], \'image\': clip[\'heads\'][\'clip_fusion\'][\'image_dim\']}:\n        raise ValueError(\'Measured feature dimensions disagree with trained fusion checkpoint\')\n    normalize = lambda title: \'\'.join(c for c in title.casefold() if c.isalnum())\n    if any(normalize(title) == normalize(FARGO_TITLE) for frame in frames.values() for title in frame.clean_title):\n        raise ValueError(\'Fargo diagnostic appears in a dataset split. Stop; do not silently drop or train on it.\')\n    if fargo_image is not None:\n        if not Path(fargo_image).is_file():\n            raise FileNotFoundError(f\'Missing Fargo image: {fargo_image}\')\n        fargo_sha = sha_file(fargo_image)\n        if any(fargo_sha in set(frame.image_sha256) for frame in frames.values()):\n            raise ValueError(\'Fargo diagnostic image is in the dataset; stop without modifying splits\')\n    return data, base, clip, dims\n\n\ndef existing_heads(baseline_bundle, clip_bundle):\n    base = json.loads((Path(baseline_bundle) / \'config.json\').read_text())\n    clip = json.loads((Path(clip_bundle) / \'config.json\').read_text())\n    result = {}\n    path = Path(baseline_bundle) / \'models/text.pt\'\n    expected_sha = clip[\'provenance\'][\'baseline\'][\'checkpoint_sha256\'][\'text\']\n    if sha_file(path) != expected_sha:\n        raise ValueError(\'Text checkpoint differs from the existing experiment\')\n    saved = torch.load(path, map_location=\'cpu\', weights_only=True)\n    if saved[\'run_id\'] != base[\'run_id\'] or saved[\'kind\'] != \'text\':\n        raise ValueError(\'Wrong text checkpoint\')\n    head = Head(\'text\'); head.load_state_dict(saved[\'state_dict\']); result[\'text\'] = head.eval().requires_grad_(False)\n    for name in (\'clip_image\', \'clip_fusion\'):\n        saved = torch.load(Path(clip_bundle) / \'models\' / f\'{name}.pt\', map_location=\'cpu\', weights_only=True)\n        if saved[\'run_id\'] != clip[\'run_id\'] or saved[\'spec\'] != clip[\'heads\'][name]:\n            raise ValueError(\'Wrong CLIP checkpoint\')\n        head = ClipHead(**saved[\'spec\']); head.load_state_dict(saved[\'state_dict\']); result[name] = head.eval().requires_grad_(False)\n    return result\n\n\ndef verify_saved_predictions(data, heads, clip_bundle, split, device):\n    expected = pd.read_csv(Path(clip_bundle) / \'results\' / f\'{split}_predictions.csv\', dtype={\'id\': str}).set_index(\'id\')\n    if len(expected) != len(data[\'ids\']) or set(expected.index) != set(data[\'ids\']):\n        raise ValueError(\'Saved predictions have different IDs\')\n    expected = expected.loc[data[\'ids\']]\n    if expected.label.tolist() != data[\'labels\'].tolist():\n        raise ValueError(\'Saved prediction labels differ\')\n    result = {}\n    for name, model in heads.items():\n        fake, _ = forward_batches(model, data, device)\n        column = (\'distilbert\' if name == \'text\' else name) + \'_fake_probability\'\n        if not np.allclose(fake, expected[column], atol=.002, rtol=.002):\n            raise ValueError(f\'{split}/{name} cannot reproduce saved probabilities; do not train on these caches\')\n        result[name] = fake\n    return result\n\n\ndef train_seed(data, run, config, seed, run_id):\n    device = gpu()\n    if config[\'epochs\'] < 1 or config[\'patience\'] < 1 or config[\'batch_size\'] < 1:\n        raise ValueError(\'Epochs, patience and batch size must be positive\')\n    spec = dict(text_dim=data[\'train\'][\'text\'].shape[1], image_dim=data[\'train\'][\'image\'].shape[1], branch_dropout=.2, head_dropout=.3)\n    contract = {\'run_id\': run_id, \'seed\': seed, \'config\': config, \'spec\': spec}\n    trial_id = digest(contract)\n    folder = Path(run) / \'seeds\' / str(seed)\n    recovery = folder / \'recovery.pt\'\n    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True\n    model = GatedFusion(**spec).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config[\'lr\'], weight_decay=config[\'weight_decay\'])\n    start, stale, best_score, best_loss, best_state, history = 0, 0, -1., float(\'inf\'), None, []\n    if recovery.exists():\n        saved = torch.load(recovery, map_location=\'cpu\', weights_only=True)\n        if saved[\'trial_id\'] != trial_id:\n            raise ValueError(\'Recovery configuration changed; use its distinct run directory\')\n        model.load_state_dict(saved[\'state_dict\']); optimizer.load_state_dict(saved[\'optimizer\'])\n        start, stale, best_score, best_loss = saved[\'epoch\'] + 1, saved[\'stale\'], saved[\'best_score\'], saved[\'best_loss\']\n        best_state, history = saved[\'best_state\'], saved[\'history\']\n        torch.set_rng_state(saved[\'rng\']); torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n    train = {k: data[\'train\'][k].to(device) for k in (\'text\', \'image\', \'labels\')}\n    val = {k: data[\'validate\'][k].to(device) for k in (\'text\', \'image\', \'labels\')}\n    criterion = nn.CrossEntropyLoss()\n    for epoch in range(start, config[\'epochs\']):\n        if stale >= config[\'patience\']:\n            break\n        model.train(); order = torch.randperm(len(train[\'labels\']), device=device); total = 0.\n        for ids in order.split(config[\'batch_size\']):\n            optimizer.zero_grad(set_to_none=True)\n            logits, _, _ = model(train[\'text\'][ids], train[\'image\'][ids])\n            loss = criterion(logits, train[\'labels\'][ids])  # Raw logits, no softmax before loss.\n            loss.backward(); optimizer.step(); total += loss.item() * len(ids)\n        model.eval()\n        with torch.inference_mode():\n            logits, gt, gi = model(val[\'text\'], val[\'image\'])\n            val_loss = criterion(logits, val[\'labels\']).item()\n            m = metrics(val[\'labels\'].cpu().numpy(), logits.softmax(1)[:, 0].cpu().numpy())\n            gates = torch.stack([gt, gi], 1).cpu().numpy()\n        if not np.isfinite(val_loss):\n            raise ValueError(\'Non-finite validation loss\')\n        score = m[\'macro_f1\']\n        if (score, -val_loss) > (best_score, -best_loss):\n            best_score, best_loss, stale = score, val_loss, 0\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            stale += 1\n        history.append({\'epoch\': epoch + 1, \'train_loss\': total / len(order), \'validation_loss\': val_loss,\n                        \'validation_metrics\': m, \'validation_gate_statistics\': gate_statistics(gates)})\n        atomic_save({\'trial_id\': trial_id, \'epoch\': epoch, \'stale\': stale, \'best_score\': best_score, \'best_loss\': best_loss,\n                     \'best_state\': best_state, \'history\': history, \'state_dict\': model.state_dict(),\n                     \'optimizer\': optimizer.state_dict(), \'rng\': torch.get_rng_state(), \'cuda_rng\': torch.cuda.get_rng_state_all()}, recovery)\n        print(f\'seed={seed} epoch={epoch + 1} validation macro-F1={score:.4f}\')\n    if best_state is None:\n        raise ValueError(\'No completed epoch\')\n    saved = {**contract, \'trial_id\': trial_id, \'labels\': {\'0\': \'Fake\', \'1\': \'Real\'}, \'state_dict\': best_state,\n             \'validation_macro_f1\': best_score, \'validation_loss\': best_loss}\n    atomic_save(saved, folder / \'best.pt\'); save_json(history, folder / \'history.json\')\n    return saved\n\n\ndef select_seeds(data, run, config, seeds, run_id):\n    if not seeds or len(seeds) != len(set(seeds)):\n        raise ValueError(\'Supply distinct random seeds\')\n    records = [train_seed({s: data[s] for s in (\'train\', \'validate\')}, run, config, seed, run_id) for seed in seeds]\n    chosen = max(records, key=lambda r: (r[\'validation_macro_f1\'], -r[\'validation_loss\']))\n    atomic_save(chosen, Path(run) / \'gated_fusion.pt\')\n    values = [r[\'validation_macro_f1\'] for r in records]\n    selection = {\'criterion\': \'Highest validation macro-F1, then lowest validation loss, then seed list order. No test or Fargo selection.\',\n                 \'selected_seed\': chosen[\'seed\'], \'selected_trial_id\': chosen[\'trial_id\'],\n                 \'validation_macro_f1_mean\': float(np.mean(values)), \'validation_macro_f1_std\': float(np.std(values)),\n                 \'seeds\': [{k: r[k] for k in (\'seed\', \'validation_macro_f1\', \'validation_loss\')} for r in records]}\n    save_json(selection, Path(run) / \'selection.json\')\n    model = GatedFusion(**chosen[\'spec\']); model.load_state_dict(chosen[\'state_dict\'])\n    return model.eval(), selection\n\n\ndef evaluate_selected(model, data, original, threshold, run, split, device):\n    run = Path(run); output = run / \'results\' / split; output.mkdir(parents=True, exist_ok=True)\n    fake, gates = forward_batches(model, data, device)\n    disagreement = disagreement_report(data[\'labels\'].numpy(), original[\'text\'], original[\'clip_image\'], fake, threshold, split)\n    report = {\'split\': split, \'gated\': metrics(data[\'labels\'].numpy(), fake),\n              \'concatenation\': metrics(data[\'labels\'].numpy(), original[\'clip_fusion\']),\n              \'gates\': gate_statistics(gates), \'disagreement\': disagreement}\n    save_json(report, output / \'metrics.json\')\n    pd.DataFrame({\'id\': data[\'ids\'], \'label\': data[\'labels\'].tolist(), \'text_fake\': original[\'text\'],\n                  \'clip_image_fake\': original[\'clip_image\'], \'concatenation_fake\': original[\'clip_fusion\'],\n                  \'gated_fake\': fake, \'g_text\': gates[:, 0], \'g_image\': gates[:, 1],\n                  \'disagreement\': np.abs(original[\'text\'] - original[\'clip_image\'])}).to_csv(output / \'predictions.csv\', index=False)\n    selection = json.loads((run / \'selection.json\').read_text())\n    history = json.loads((run / \'seeds\' / str(selection[\'selected_seed\']) / \'history.json\').read_text())\n    diagnostic_plots(data[\'labels\'].numpy(), {\'Concatenation\': original[\'clip_fusion\'], \'Gated\': fake}, gates, history, disagreement, output)\n    if split == \'validate\':\n        save_json(warning_policy(disagreement), run / \'warning_policy.json\')\n    save_json({\'complete\': True, \'split\': split}, output / \'complete.json\')\n    return report\n', 'gated_inference.py': '"""Independent Windows CPU inference. Instantiate once and reuse across predictions."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport torch\nfrom PIL import Image\nfrom clip_models import ClipEncoder, CachedTextEncoder, ClipHead, FEATURE_SPEC\nfrom models import Head\nfrom gated_model import GatedFusion\n\n\nclass GatedPredictor:\n    def __init__(self, bundle):\n        bundle = Path(bundle)\n        self.config = json.loads((bundle / \'config.json\').read_text())\n        if self.config[\'labels\'] != {\'0\': \'Fake\', \'1\': \'Real\'} or self.config[\'feature_spec\'] != FEATURE_SPEC:\n            raise ValueError(\'Incompatible label mapping or preprocessing\')\n        manifest = json.loads((bundle / \'sha256.json\').read_text())\n        for name, expected in manifest.items():\n            path = (bundle / name).resolve()\n            if not path.is_relative_to(bundle.resolve()) or not path.is_file():\n                raise FileNotFoundError(f\'Missing or unsafe bundle artifact: {name}\')\n            if hashlib.sha256(path.read_bytes()).hexdigest() != expected:\n                raise ValueError(f\'Bundle checksum mismatch: {name}\')\n        saved = torch.load(bundle / \'gated_fusion.pt\', map_location=\'cpu\', weights_only=True)\n        if saved[\'run_id\'] != self.config[\'run_id\'] or saved[\'spec\'] != self.config[\'spec\']:\n            raise ValueError(\'Gated checkpoint/configuration mismatch\')\n        self.gated = GatedFusion(**saved[\'spec\']).eval().requires_grad_(False)\n        self.gated.load_state_dict(saved[\'state_dict\'])\n        self.text = CachedTextEncoder(bundle / \'encoders/distilbert\', device=\'cpu\')\n        self.image = ClipEncoder(bundle / \'encoders/clip\', device=\'cpu\', local=True)\n        self.heads = {}\n        for name in (\'text\', \'clip_image\', \'clip_fusion\'):\n            state = torch.load(bundle / \'models\' / (name + \'.pt\'), map_location=\'cpu\', weights_only=True)\n            model = Head(\'text\') if name == \'text\' else ClipHead(**state[\'spec\'])\n            model.load_state_dict(state[\'state_dict\'])\n            self.heads[name] = model.eval().requires_grad_(False)\n\n    @torch.inference_mode()\n    def predict(self, title, image):\n        if not title.strip():\n            raise ValueError(\'Text must be nonempty\')\n        text, visual = self.text.encode([title.strip()]), self.image.encode([image])\n        logits, gt, gi = self.gated(text, visual)\n        probabilities = {name: model(text, visual).softmax(1)[0].tolist() for name, model in self.heads.items()}\n        probabilities[\'gated_fusion\'] = logits.softmax(1)[0].tolist()\n        disagreement = abs(probabilities[\'text\'][0] - probabilities[\'clip_image\'][0])\n        policy = self.config[\'warning_policy\']\n        return {\'probabilities_order\': [\'Fake\', \'Real\'], \'probabilities\': probabilities,\n                \'label\': self.config[\'labels\'][str(int(logits.argmax(1).item()))],\n                \'g_text\': gt.item(), \'g_image\': gi.item(), \'disagreement\': disagreement,\n                \'warning\': policy[\'message\'] if policy[\'enabled\'] and disagreement >= policy[\'threshold\'] else None,\n                \'gate_note\': \'Learned coefficients are not definitive feature importance.\'}\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--bundle\', default=\'gated_artifacts\')\n    parser.add_argument(\'--text\', required=True)\n    parser.add_argument(\'--image\', required=True)\n    args = parser.parse_args()\n    with Image.open(args.image) as image:\n        print(json.dumps(GatedPredictor(args.bundle).predict(args.text, image), indent=2))\n', 'gated_export.py': '"""Export separate real weights and verify CPU predictions before declaring deployment ready."""\nimport json\nimport shutil\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom clip_experiment import sha_file, guard_output\nfrom gated_analysis import save_json, forward_batches\n\n\ndef export_bundle(run, baseline, clip, text_assets, image_assets, source_dir):\n    run, baseline, clip = map(Path, (run, baseline, clip))\n    destination = run / \'windows_export\'\n    guard_output(destination, baseline, clip, text_assets, image_assets)\n    config = json.loads((clip / \'config.json\').read_text())\n    # The original tokenizer and text weights must be identical, not merely the same model name.\n    for name, expected in config[\'provenance\'][\'baseline\'][\'text_asset_sha256\'].items():\n        path = Path(text_assets) / name\n        if not path.is_file() or sha_file(path) != expected:\n            raise ValueError(f\'Missing/different original DistilBERT asset: {path}\')\n    for folder, filenames in [(image_assets, [\'config.json\', \'preprocessor_config.json\']), (text_assets, [\'config.json\'])]:\n        for name in filenames:\n            if not (Path(folder) / name).is_file():\n                raise FileNotFoundError(Path(folder) / name)\n    bundle = destination / \'gated_artifacts\'\n    bundle.mkdir(parents=True, exist_ok=True)\n    (bundle / \'models\').mkdir(exist_ok=True)\n    shutil.copy2(run / \'gated_fusion.pt\', bundle / \'gated_fusion.pt\')\n    shutil.copy2(baseline / \'models/text.pt\', bundle / \'models/text.pt\')\n    for name in (\'clip_image\', \'clip_fusion\'):\n        shutil.copy2(clip / \'models\' / (name + \'.pt\'), bundle / \'models\' / (name + \'.pt\'))\n    for kind, folder in [(\'distilbert\', text_assets), (\'clip\', image_assets)]:\n        shutil.copytree(folder, bundle / \'encoders\' / kind, dirs_exist_ok=True)\n    checkpoint = torch.load(run / \'gated_fusion.pt\', map_location=\'cpu\', weights_only=True)\n    export_config = {k: checkpoint[k] for k in [\'run_id\', \'spec\', \'seed\', \'labels\', \'config\']}\n    export_config.update(feature_spec=config[\'feature_spec\'], text_encoder=config[\'text_encoder\'],\n                         warning_policy=json.loads((run / \'warning_policy.json\').read_text()),\n                         baseline_run_id=config[\'baseline_run_id\'], clip_run_id=config[\'run_id\'])\n    save_json(export_config, bundle / \'config.json\')\n    for name in (\'selection.json\', \'experiment_config.json\'):\n        shutil.copy2(run / name, bundle / name)\n    shutil.copytree(run / \'results\', bundle / \'results\', dirs_exist_ok=True)\n    for name in (\'gated_inference.py\', \'gated_model.py\', \'clip_models.py\', \'models.py\'):\n        shutil.copy2(Path(source_dir) / name, destination / name)\n    (destination / \'requirements.txt\').write_text(\'torch\\ntorchvision\\ntransformers==4.57.1\\nPillow\\nsafetensors\\n\', encoding=\'utf-8\')\n    save_json({p.relative_to(bundle).as_posix(): sha_file(p) for p in bundle.rglob(\'*\') if p.is_file() and p.name != \'sha256.json\'}, bundle / \'sha256.json\')\n    return destination\n\n\ndef cpu_smoke(export_dir, frame, cached, model, output):\n    from gated_inference import GatedPredictor\n    start = time.perf_counter()\n    predictor = GatedPredictor(Path(export_dir) / \'gated_artifacts\')\n    cold_load = time.perf_counter() - start\n    rows = []\n    # First three validation samples are predeclared; no cherry-picking or test selection.\n    for index in range(min(3, len(frame))):\n        row = frame.iloc[index]\n        inputs = {k: cached[k][index:index+1] for k in (\'text\', \'image\')}\n        expected, gates = forward_batches(model, inputs, \'cpu\')\n        predictions, seconds = [], []\n        with Image.open(row.resolved_path) as image:\n            with torch.inference_mode():\n                raw_text = predictor.text.encode([row.clean_title])\n                raw_image = predictor.image.encode([image])\n            for kind, actual_feature in [(\'text\', raw_text), (\'image\', raw_image)]:\n                if not torch.allclose(actual_feature, inputs[kind], atol=.002, rtol=.002):\n                    raise ValueError(f\'Exported {kind} encoder does not reproduce original cached features\')\n            for repeat in range(2):\n                start = time.perf_counter(); predictions.append(predictor.predict(row.clean_title, image)); seconds.append(time.perf_counter() - start)\n        actual = predictions[0]\n        with torch.inference_mode():\n            for name, head in predictor.heads.items():\n                expected_reference = head(inputs[\'text\'], inputs[\'image\']).softmax(1)[0].numpy()\n                if not np.allclose(actual[\'probabilities\'][name], expected_reference, atol=.002, rtol=.002):\n                    raise ValueError(f\'Exported {name} baseline parity failed\')\n        if actual != predictions[1]:\n            raise ValueError(\'Repeated CPU inference is inconsistent\')\n        if not np.allclose(actual[\'probabilities\'][\'gated_fusion\'][0], expected[0], atol=.002, rtol=.002):\n            raise ValueError(\'Raw-input CPU inference differs from cached-feature predictions\')\n        if not np.allclose([actual[\'g_text\'], actual[\'g_image\']], gates[0], atol=.002, rtol=.002):\n            raise ValueError(\'CPU gate mismatch\')\n        rows.append({\'id\': row.id, \'prediction\': actual, \'cached_fake\': float(expected[0]), \'seconds\': seconds})\n    save_json({\'passed\': True, \'cold_load_seconds\': cold_load, \'samples\': rows}, output)\n    return predictor\n'}
for name, source in SOURCE_FILES.items():
    (WORK / name).write_text(source, encoding='utf-8')


## 2. Paths and predeclared settings
Point BASELINE and CLIP at the **existing run directories** (or exported bundles with config.json, models/, results/). Cache roots are their original **features** directories, not Windows exports. Uploading a dataset ZIP does not restore missing caches.
For CPU export, set TEXT_ASSETS and CLIP_ASSETS to the corresponding existing encoder folders. A complete existing clip_artifacts export contains both. These are checked at export; head training needs only caches.
Do not change these settings after inspecting test results. The optional Fargo path can be supplied now solely to check exclusion, or uploaded at the final diagnostic cell.


In [ ]:
DRIVE = Path('/content/drive/MyDrive')
BASELINE = DRIVE / 'Fakeddit_Demo/runs/27751e2dbfb636fc5929'
CLIP = DRIVE / 'Fakeddit_CLIP_Experiment/runs/29037a54744b05c6dc32'
TEXT_CACHE = DRIVE / 'Fakeddit_Demo/features'
CLIP_CACHE = DRIVE / 'Fakeddit_CLIP_Experiment/features'
DATA_DIR = DRIVE / 'Multimodal_Fake_News_Detection/data'
DATA_SOURCE = 'drive'  # 'drive' or 'zip'
ZIP_ON_DRIVE = ''  # Existing processed dataset ZIP; blank prompts upload in zip mode.
TEXT_ASSETS = DRIVE / 'Fakeddit_Demo/encoders_v1/distilbert'
# Set this to the original encoder folder or an uploaded clip_artifacts/encoders/clip.
CLIP_ASSETS = DRIVE / 'clip_artifacts/encoders/clip'
FARGO_IMAGE = ''
OUTPUT = DRIVE / 'Fakeddit_Gated_Fusion'
TRAINING = dict(lr=.0003, batch_size=64, epochs=30, weight_decay=.001, patience=5)
SEEDS = [42, 43, 44]  # Choose before training. Keep unchanged when resuming.
DISAGREEMENT_QUANTILE = .8
FIXED_DISAGREEMENT_THRESHOLD = None  # Optional predeclared threshold in [0,1].
ABLATION_SEED, ABLATION_REPETITIONS = 42, 5
from dataset import validate, extract_zip
from clip_experiment import guard_output, digest, sha_file
from gated_analysis import *
from gated_pipeline import *
guard_output(OUTPUT, BASELINE, CLIP, TEXT_CACHE, CLIP_CACHE, DATA_DIR, TEXT_ASSETS, CLIP_ASSETS)
OUTPUT.mkdir(parents=True, exist_ok=True)
if DATA_SOURCE == 'zip':
    if ZIP_ON_DRIVE:
        archive = Path(ZIP_ON_DRIVE)
    else:
        uploaded = files.upload()
        assert len(uploaded) == 1, 'Upload exactly one existing processed dataset ZIP.'
        archive = WORK / next(iter(uploaded))
        del uploaded
    DATA_DIR = extract_zip(archive, Path('/content/fakeddit_gated_data'))
elif DATA_SOURCE != 'drive':
    raise ValueError('DATA_SOURCE must be drive or zip')
guard_output(OUTPUT, DATA_DIR)
for folder in (BASELINE, CLIP):
    assert (folder / 'config.json').is_file(), f'Missing existing run configuration: {folder}/config.json'


## 3. Verify original data, labels, sample alignment and caches
Validate every image, preserve CSVs, resolve Windows paths on Linux, join feature shards by ID, verify labels and text cache hashes. CLIP feature IDs and measured dimensions must match the original trained heads. No rows are dropped.


In [ ]:
frames, dataset_report = validate(DATA_DIR, OUTPUT / 'dataset_validation.json')
data, base_config, clip_config, dimensions = load_caches(frames, BASELINE, CLIP, TEXT_CACHE, CLIP_CACHE, FARGO_IMAGE or None)
heads = existing_heads(BASELINE, CLIP)
cache_hashes = {s: {k: hashlib.sha256(d[k].contiguous().numpy().tobytes()).hexdigest() for k in ('text','image')} for s,d in data.items()}
contract = dict(training=TRAINING, seeds=SEEDS, dimensions=dimensions, labels=base_config['labels'],
    baseline_run_id=base_config['run_id'], clip_run_id=clip_config['run_id'], cache_hashes=cache_hashes,
    text_encoder=base_config['encoders'], feature_spec=clip_config['feature_spec'],
    source_sha256={name: hashlib.sha256(source.encode()).hexdigest() for name,source in SOURCE_FILES.items()},
    threshold_quantile=DISAGREEMENT_QUANTILE, fixed_threshold=FIXED_DISAGREEMENT_THRESHOLD,
    ablation_seed=ABLATION_SEED, ablation_repetitions=ABLATION_REPETITIONS)
RUN_ID = digest(contract)
RUN = OUTPUT / 'runs' / RUN_ID
RUN.mkdir(parents=True, exist_ok=True)
save_json(contract, RUN / 'experiment_config.json')
original_validation = verify_saved_predictions(data['validate'], heads, CLIP, 'validate', 'cuda')
print('Verified:', dimensions, {s:len(d['ids']) for s,d in data.items()}, RUN)


## 4. Original concatenation ablations — before creating/training the gated model
Five label-blind derangements, training-mean replacements and Gaussian perturbations. Save per-ID changes, class flips, plots and a descriptive bootstrap interval. Intervention accuracy is omitted: mismatched pairs do not have justified labels. These diagnostics cannot prove causal importance or that a modality is ignored.


In [ ]:
baseline_ablation = ablations(heads['clip_fusion'], data['validate'], data['train'], RUN / 'ablations/concatenation_validate',
    'cuda', seed=ABLATION_SEED, repetitions=ABLATION_REPETITIONS)
print(json.dumps(baseline_ablation, indent=2))
threshold = choose_threshold(original_validation['text'], original_validation['clip_image'],
    quantile=DISAGREEMENT_QUANTILE, fixed=FIXED_DISAGREEMENT_THRESHOLD)
save_json(threshold, RUN / 'threshold.json')


## 5. Train only the gated head; recover completed epochs and seeds
Each modality projects to 256 with LayerNorm, ReLU and dropout .2. A 512→128→2 softmax gate weights the branches. Their concatenation feeds a 512→128→2 head with dropout .3. Cross-entropy receives raw logits. Encoders are never involved in training.
AdamW, validation macro-F1 selection, patience 5. Save optimizer, CPU/CUDA RNG, history, best weights every epoch. Rerun from the top after disconnecting with unchanged settings: completed epochs/seeds are recovered. No feature extraction repeats because this notebook never extracts caches.


In [ ]:
assert (RUN / 'ablations/concatenation_validate/report.json').is_file()
gated, selection = select_seeds(data, RUN, TRAINING, SEEDS, RUN_ID)
validation = evaluate_selected(gated, data['validate'], original_validation, threshold, RUN, 'validate', 'cuda')
gated_ablation = ablations(gated, data['validate'], data['train'], RUN / 'ablations/gated_validate',
    'cuda', seed=ABLATION_SEED, repetitions=ABLATION_REPETITIONS)
improved = validation['gated']['macro_f1'] > validation['concatenation']['macro_f1']
save_json(dict(validation_improved=improved, selection=selection, threshold=threshold,
    decision='Frozen before held-out evaluation; no automatic app promotion'), RUN / 'final_selection.json')
print('Validation macro-F1:', {k:validation[k]['macro_f1'] for k in ('concatenation','gated')})
print('Validation improved:', improved, '\nSeed summary:', selection)


## 6. Held-out evaluation of the final selected model
Do not tune settings after this cell. Completed test results are reused on recovery. Test data never select epochs, seeds, thresholds or warning policy.


In [ ]:
import pandas as pd
assert (RUN / 'final_selection.json').is_file()
test_path = RUN / 'results/test/metrics.json'
if test_path.exists() and (test_path.parent / 'complete.json').exists():
    test = json.loads(test_path.read_text())
else:
    original_test = verify_saved_predictions(data['test'], heads, CLIP, 'test', 'cuda')
    test = evaluate_selected(gated, data['test'], original_test, threshold, RUN, 'test', 'cuda')
comparison = pd.DataFrame([dict(model=name, validation_macro_f1=validation[key]['macro_f1'],
    test_accuracy=test[key]['accuracy'], test_macro_f1=test[key]['macro_f1'], test_roc_auc=test[key]['roc_auc'])
    for name,key in [('DistilBERT + CLIP concatenation','concatenation'),('DistilBERT + CLIP gated','gated')]])
comparison.to_csv(RUN / 'comparison.csv', index=False)
display(comparison)
report = dict(validation=validation, test=test, validation_improved=improved,
    baseline_ablation=baseline_ablation, gated_ablation=gated_ablation,
    interpretation='Compare measured swap sensitivity, not just gate values. OOD interventions do not establish causal importance or robustness to misinformation.',
    deployment='Current application unchanged; CPU export verification required.')
save_json(report, RUN / 'evaluation_report.json')
from IPython.display import display, Image as DisplayImage
for path in [RUN/'ablations/concatenation_validate/ablation.png', RUN/'ablations/gated_validate/ablation.png', *sorted((RUN/'results/test').glob('*.png'))]:
    display(DisplayImage(filename=str(path)))


## 7. Separate Windows CPU export and raw-input parity checks
Supply the **original** frozen encoder assets. Missing assets stop export. Check three predeclared validation examples against cached-feature output and require identical repeated CPU predictions. Cold-load and inference timings are saved. A ZIP is created only after these checks pass; the app remains unchanged.


In [ ]:
from gated_export import export_bundle, cpu_smoke
export_dir = export_bundle(RUN, BASELINE, CLIP, TEXT_ASSETS, CLIP_ASSETS, WORK)
predictor = cpu_smoke(export_dir, frames['validate'], data['validate'], gated, RUN / 'cpu_verification.json')
shutil.copy2(RUN / 'cpu_verification.json', export_dir / 'cpu_verification.json')
archive = shutil.make_archive(str(RUN / 'gated_windows_export'), 'zip', root_dir=export_dir)
print('Verified CPU bundle:', archive)
print('Download from Drive; keep separate from the current application.')


## 8. Optional Fargo qualitative diagnostic — after selection and evaluation
Set FARGO_IMAGE above to the existing image, or upload **flamethrower.jpg** here. Never adjust settings in response to this result. User supplies the Fake label; this is not part of benchmark performance. Gate values are learned coefficients, not definitive feature importance.


In [ ]:
RUN_FARGO = False  # Set True to upload/run the diagnostic after completing cells above.
if RUN_FARGO:
    if not FARGO_IMAGE:
        uploaded = files.upload()
        assert len(uploaded) == 1, 'Upload the supplied flamethrower image only.'
        FARGO_IMAGE = str(WORK / next(iter(uploaded)))
        del uploaded
    assert not any(sha_file(FARGO_IMAGE) in set(frame.image_sha256) for frame in frames.values()), 'Diagnostic appears in data; stop.'
    from PIL import Image
    with Image.open(FARGO_IMAGE) as image:
        result = predictor.predict(FARGO_TITLE, image)
    save_json(dict(headline=FARGO_TITLE, label=0, label_source='User supplied', qualitative_only=True,
        image_sha256=sha_file(FARGO_IMAGE), measured=result,
        previously_reported_fake=dict(text=.004,resnet_image=.75,resnet_fusion=.163,clip_image=.907,clip_fusion=.06)), RUN / 'fargo_diagnostic.json')
    print(json.dumps(result, indent=2))
else:
    print('Fargo diagnostic skipped; benchmark results are unaffected.')


## 9. What to retain and what to do next
Drive run folder contains the actual trained `gated_fusion.pt`, `selection.json`, configuration, epoch recovery files, evaluation report, comparison CSV, plots, intervention tables, threshold/warning policy and verified Windows ZIP.
No trained gated checkpoint is supplied with this notebook before execution. Missing caches must be restored from the original Drive experiment; do not substitute Windows checkpoints for feature shards.
Only after inspecting **validation** improvement and CPU parity, create a separate app copy to add an optional sixth model. Instantiate/cache `GatedPredictor` once; reuse it for predictions. Display warnings without overriding labels. Keep CLIP fusion as default unless promotion is justified. The existing application requires no rollback because this workflow never changes it.
